# Averaged across problems (≥ cutoff): Rank Proportions around good reversals

This notebook reproduces your per-problem analysis:

- `get_good_reversal_info(..., include_first_block=True)`
- `get_rank_counts_by_good_reversal(reversal_windows)`
- `pvalue_paired_t_best_vs_second_vs_third(rank_counts_by_good_reversal)`
- `plot_rank_proportions(...)`

…but instead of generating one figure per problem, it **averages across problems ≥ `CUTOFF_PROBLEM`**.

### How averaging works here
`rank_counts_by_good_reversal` is a dict keyed by mouse/subject, containing per-reversal aligned rank-proportion/count data.
To average across problems, we **concatenate the per-problem reversal windows per subject** (i.e., treat each reversal instance as another sample),
then pass the pooled structure into `plot_rank_proportions` and the p-value function.

This is safe because rank counts are computed per reversal window and don't rely on global trial indices across problems.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from src.behavior_import.import_data import *
from src.behavior_import.extract_trials import *
from src.behavior_analysis.get_good_reversal_info import *
from src.behavior_analysis.get_rank_counts_by_good_reversal import *
from src.behavior_analysis.get_diagnostic_p_value import *
from src.behavior_visualization.plot_rank_proportions import *
from fix_grid_maze_cohort_02_problems import *

import numpy as np


## Parameters

In [2]:
task = "grid-maze"
# task = "open-field"

CUTOFF_PROBLEM = 7  # average problems >= this

OUT_BASE = Path("../results/figures")


## Load data and group into problems

In [3]:
folder_name = None
cohort = None
if task == "grid-maze":
    cohort = "cohort-02"
    folder_name = "3x3_maze_blocked_reward_bandit"
elif task == "open-field":
    cohort = "cohort-01"
    folder_name = "3x3_field_blocked_reward_bandit"

root = f"/Volumes/behrens/meg/{folder_name}/{cohort}/rawdata/"

subjects_data = import_data(root)
subjects_trials_by_problem = extract_trials_grouped_by_problem(subjects_data)

if task == "grid-maze" and cohort == "cohort-02":
    subjects_trials_by_problem = fix_grid_maze_cohort_02_problems(subjects_trials_by_problem)

problem_numbers = sorted(subjects_trials_by_problem.keys())
probs_used = [p for p in problem_numbers if p >= CUTOFF_PROBLEM]
print("Problems found:", problem_numbers)
print("Averaging problems >=", CUTOFF_PROBLEM, ":", probs_used)


[WARN] Failed to read /Volumes/behrens/meg/3x3_maze_blocked_reward_bandit/cohort-02/rawdata/sub-03_id-MY_04_R/ses-27_date-20260124/behav/._MY_04_R-2026-01-24-124422.tsv: 'utf-8' codec can't decode byte 0xb0 in position 37: invalid start byte.
[WARN] Failed to read /Volumes/behrens/meg/3x3_maze_blocked_reward_bandit/cohort-02/rawdata/sub-03_id-MY_04_R/ses-55_date-20260209/behav/._MY_04_R-2026-02-09-103918.tsv: 'utf-8' codec can't decode byte 0xb0 in position 37: invalid start byte.
[WARN] Failed to read /Volumes/behrens/meg/3x3_maze_blocked_reward_bandit/cohort-02/rawdata/sub-03_id-MY_04_R/ses-28_date-20260124/behav/._MY_04_R-2026-01-24-165301.tsv: 'utf-8' codec can't decode byte 0xb0 in position 37: invalid start byte.
[WARN] Failed to read /Volumes/behrens/meg/3x3_maze_blocked_reward_bandit/cohort-02/rawdata/sub-03_id-MY_04_R/ses-12_date-20260116/behav/._MY_04_R-2026-01-16-143640.tsv: 'utf-8' codec can't decode byte 0xb0 in position 37: invalid start byte.
[WARN] Failed to read /Volum

## Build and merge reversal windows across problems ≥ cutoff

In [4]:
from src.behavior_analysis.merge_across_problems import merge_windows_by_subject

windows_by_problem = {}
for p in sorted(subjects_trials_by_problem.keys()):
    if p < CUTOFF_PROBLEM:
        continue
    subjects_trials = subjects_trials_by_problem[p]
    rw = get_good_reversal_info(subjects_trials, include_first_block=True)
    windows_by_problem[p] = rw

reversal_windows_merged = merge_windows_by_subject(windows_by_problem)

print("Merged subjects:", len(reversal_windows_merged))
print("Total merged reversals:", sum(len(v) for v in reversal_windows_merged.values()))

Merged subjects: 6
Total merged reversals: 710


## Rank counts + p-values + plot (averaged across problems)

In [5]:
import src.behavior_visualization.plot_style  # noqa: F401 — applies rcParams
from src.behavior_visualization.plot_problem_summary import (
    plot_rank_proportions_average_without_blocks,
)

In [6]:
rank_counts_by_good_reversal = get_rank_counts_by_good_reversal(reversal_windows_merged)
p_values = pvalue_paired_t_best_vs_second_vs_third(rank_counts_by_good_reversal)

save_path = OUT_BASE / task / cohort / f"problems-geq-{CUTOFF_PROBLEM:02d}" / "choice-stats" / "Rank Proportions (Averaged)"
plot_rank_proportions_average_without_blocks(rank_counts_by_good_reversal, average_across_mice_pvalues=p_values, save_path=str(save_path))
